# Stock Markets Analytics Zoomcamp 2026
## Homework 1: Intro and Data Sources

Notebook solution for Module 1 Homework of the **Stock Markets Analytics Zoomcamp (2026 Cohort)**.


In [1]:
import io
import requests
import pandas as pd
import numpy as np
import yfinance as yf

# Configure display options
pd.set_option('display.max_columns', None)


---
### Question 1: [Index] S&P 500 Stocks Added to the Index

**Question:** Which year had the highest number of additions (starting from 2020)?  
**Options:** 2025, 2024, 2023, 2022


In [2]:
# 1. Scrape S&P 500 table from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)
sp500_df = pd.read_html(io.StringIO(response.text))[0]

# 2. Extract year of addition
sp500_df['Year'] = pd.to_datetime(sp500_df['Date added'], errors='coerce').dt.year

# 3. Additions starting from 2020
additions = sp500_df[sp500_df['Year'] >= 2020]['Year'].value_counts().sort_index()
print("S&P 500 additions per year (from 2020):")
print(additions)

best_year = int(additions.idxmax())
print(f"\n-> Answer Q1: {best_year}")


S&P 500 additions per year (from 2020):
Year
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
2026    13
Name: count, dtype: int64

-> Answer Q1: 2025


---
### Question 2: [Macro] Indexes YTD (as of 21 August 2026)

**Question:** How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?  
**Options:** 1, 2, 3, 4


In [3]:
indices = {
    'United States (^GSPC)': '^GSPC',
    'China (000001.SS)': '000001.SS',
    'Hong Kong (^HSI)': '^HSI',
    'Australia (^AXJO)': '^AXJO',
    'India (^NSEI)': '^NSEI',
    'Canada (^GSPTSE)': '^GSPTSE',
    'Germany (^GDAXI)': '^GDAXI',
    'United Kingdom (^FTSE)': '^FTSE',
    'Japan (^N225)': '^N225',
    'Mexico (^MXX)': '^MXX',
    'Brazil (^BVSP)': '^BVSP'
}

returns = {}

for name, ticker in indices.items():
    # Fetch data up to 2026-08-21 inclusive
    df = yf.download(ticker, start='2026-01-01', end='2026-08-22', progress=False)
    close = df['Close']
    if isinstance(close, pd.DataFrame):
        close = close.squeeze()
        
    start_close = close.iloc[0]
    end_close = close.iloc[-1]
    returns[name] = (end_close / start_close - 1) * 100

summary_df = pd.DataFrame(list(returns.items()), columns=['Index', 'YTD Return (%)'])
us_return = returns['United States (^GSPC)']
summary_df['Beat S&P 500'] = summary_df['YTD Return (%)'] > us_return

print(summary_df.to_string(index=False))

count_beating = summary_df[(summary_df['Beat S&P 500']) & (summary_df['Index'] != 'United States (^GSPC)')].shape[0]
print(f"\n-> US S&P 500 YTD Return: {us_return:.2f}%")
print(f"-> Number of indices beating S&P 500: {count_beating}")
print(f"-> Answer Q2: {count_beating}")


                 Index  YTD Return (%)  Beat S&P 500
 United States (^GSPC)       11.896237         False
     China (000001.SS)       -2.938152         False
      Hong Kong (^HSI)       -1.249160         False
     Australia (^AXJO)        3.793632         False
         India (^NSEI)       -7.245892         False
      Canada (^GSPTSE)       14.856630          True
      Germany (^GDAXI)        6.508817         False
United Kingdom (^FTSE)        8.697531         False
         Japan (^N225)       27.364060          True
         Mexico (^MXX)        2.475501         False
        Brazil (^BVSP)        6.536106         False

-> US S&P 500 YTD Return: 11.90%
-> Number of indices beating S&P 500: 2
-> Answer Q2: 2


---
### Question 3: [Index] S&P 500 Market Corrections Analysis

**Question:** Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.  
*(Correction defined as >= 5% drop from the most recent all-time high)*  
**Options:** 8, 16, 24, 32


In [4]:
# 1. Download S&P 500 daily data from 1950
sp500 = yf.download('^GSPC', start='1950-01-01', progress=False)
close_prices = sp500['Close']
if isinstance(close_prices, pd.DataFrame):
    close_prices = close_prices.squeeze()

# 2. Identify all-time high points
cummax = close_prices.cummax()
ath_mask = (close_prices == cummax) & (close_prices > close_prices.shift(1, fill_value=0))
ath_dates = close_prices[ath_mask].index

# 3. Calculate drawdowns between consecutive ATHs
corrections = []
for i in range(len(ath_dates) - 1):
    t_start = ath_dates[i]
    t_end = ath_dates[i + 1]
    
    window = close_prices.loc[t_start:t_end]
    if len(window) > 2:
        high = window.iloc[0]
        low = window.min()
        low_date = window.idxmin()
        
        drawdown = (high - low) / high * 100
        corrections.append({
            'start_date': t_start,
            'trough_date': low_date,
            'recovery_date': t_end,
            'high': high,
            'low': low,
            'drawdown': drawdown,
            'duration_days': (low_date - t_start).days
        })

df_corr = pd.DataFrame(corrections)

# 4. Filter corrections with >= 5% drawdown
sig_corr = df_corr[df_corr['drawdown'] >= 5].copy()

median_drawdown = sig_corr['drawdown'].median()
print(f"Total corrections (>= 5%): {len(sig_corr)}")
print(f"Median drawdown: {median_drawdown:.2f}%")
print(f"25th percentile: {sig_corr['drawdown'].quantile(0.25):.2f}%")
print(f"75th percentile: {sig_corr['drawdown'].quantile(0.75):.2f}%")
print(f"\n-> Answer Q3 (nearest option): {round(median_drawdown)}")


Total corrections (>= 5%): 74
Median drawdown: 7.99%
25th percentile: 6.23%
75th percentile: 14.02%

-> Answer Q3 (nearest option): 8


---
### Question 4: [Stocks] Earnings Surprise Analysis for Amazon (AMZN)

**Question:** Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.  
**Options:** 3.35, 2.35, 1.35, 0.35


In [5]:
ticker = yf.Ticker('AMZN')

# 1. Earnings dates & surprise data
earnings = ticker.get_earnings_dates(limit=25).reset_index()
earnings['Date'] = pd.to_datetime(earnings['Earnings Date'].dt.date)

# 2. Historical price data
hist = ticker.history(start='2020-01-01')
prices = hist[['Close']].copy()
prices['Date'] = pd.to_datetime(prices.index.date)
prices = prices.reset_index(drop=True)

# 3. 2-day percentage return: Close_Day3 / Close_Day1 - 1 (Day 2 = earnings day)
prices['ret_2d'] = (prices['Close'].shift(-1) / prices['Close'].shift(1) - 1) * 100

# 4. Filter for positive earnings surprises
positive_surprises = earnings[earnings['Surprise(%)'] > 0].copy()

# 5. Merge on trading date
merged = pd.merge(positive_surprises, prices[['Date', 'ret_2d']], on='Date', how='inner')

median_2d_return = merged['ret_2d'].median()
correlation = merged['Surprise(%)'].corr(merged['ret_2d'])

print(f"Positive earnings surprises analyzed: {len(merged)}")
print(f"Median 2-day return: {median_2d_return:.4f}%")
print(f"Correlation between surprise % and 2-day return: {correlation:.4f}")
print(f"\n-> Answer Q4: {median_2d_return:.2f}")


Positive earnings surprises analyzed: 20
Median 2-day return: 0.3528%
Correlation between surprise % and 2-day return: 0.3367

-> Answer Q4: 0.35


---
### Question 5: [Exploratory, optional] Idea for your capstone project

**Project Concept: Sector Liquidity Rotation & Institutional Flow Strategy for the Vietnam Stock Market (VN-Index / VN30)**
* **Market & Asset Class:** Top 50 most liquid stocks on the Ho Chi Minh Stock Exchange (HOSE) covering key driving sectors: Banking, Securities, Real Estate, Steel, and Retail.
* **Investment Horizon:** 10 to 15 trading days (bi-weekly rebalancing), specifically aligned with Vietnam's T+2.5 settlement cycle and short-to-medium retail sentiment swings.
* **Core Methodology:**
  1. Measure sector money flow using relative liquidity shifts (Sector Daily Turnover / Total HOSE Turnover) combined with 5-day cumulative Foreign Net Trading and Domestic Proprietary Trading flow.
  2. Filter individual stocks using momentum & trend criteria (Price > 20-day SMA, 14-day RSI in the 50-65 expansion zone, and volume breakout > 1.5x 20-day volume average).
  3. Train a LightGBM classification model to estimate the probability of a stock outperforming the VN-Index over the next 10-15 trading days, coupled with dynamic trailing-stop loss rules.

---
### Question 6: [Exploratory, optional] Investigate new metrics

**Proposed Metrics for the Vietnam Stock Market:**
1. **Foreign Investor & Domestic Proprietary Net Flow (Khối ngoại & Tự doanh mua/bán ròng):**
   * *Rationale:* In an 85%+ retail-dominated market, foreign institutional funds and brokerage prop desks act as anchor price drivers and sentiment leaders. Tracking 5-day and 20-day cumulative net flow by sector helps identify institutional accumulation or distribution before broad retail trends materialize.
   * *Retrieval:* Queried via open-source Python packages (such as `vnstock` / `vnstock3`) or data feeds from SSI iBoard API / Vietstock.
2. **Sector Liquidity Concentration Ratio (% Thanh khoản ngành / Total HOSE Turnover):**
   * *Rationale:* Extreme turnover concentration (e.g. Banking or Securities consuming > 30-35% of total exchange volume) often signals either a powerful momentum breakout or an unsustainable retail FOMO climax. Monitoring shifts in this ratio helps time sector rotation early.
   * *Retrieval:* Calculated directly from daily transaction value per industry group using `vnstock` or HOSE trading statistics.
